In [2]:
#Practical Task: Graph Traversal (BFS & DFS) and Centrality Analysis

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.cm as cm

np.random.seed(42)

n_nodes = 40
G = nx.barabasi_albert_graph(n=n_nodes, m=2, seed=42)
mapping = {i: f"User_{i+1}" for i in G.nodes()}
G = nx.relabel_nodes(G, mapping)

edge_df = pd.DataFrame(G.edges(), columns=["source", "target"])
edge_df.to_csv("social_network_edges.csv", index=False)
print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Is connected: {nx.is_connected(G)}")

# Pick the highest-degree node as a natural "starting user" for traversal
start_node = max(G.degree, key=lambda x: x[1])[0]
print(f"Start node for traversal (highest degree): {start_node}")

bfs_edges = list(nx.bfs_edges(G, source=start_node))
bfs_order = [start_node] + [v for u, v in bfs_edges]
print(f"\nBFS traversal order ({len(bfs_order)} nodes reached):")
print(bfs_order)

bfs_lengths = nx.shortest_path_length(G, source=start_node)  # BFS-based shortest paths
diameter_sample = max(bfs_lengths.values())
print(f"Max BFS hop distance from {start_node}: {diameter_sample}")

# ---------------------------------------------------------------
# 3. DFS traversal
# ---------------------------------------------------------------
dfs_edges = list(nx.dfs_edges(G, source=start_node))
dfs_order = [start_node] + [v for u, v in dfs_edges]
print(f"\nDFS traversal order ({len(dfs_order)} nodes reached):")
print(dfs_order)

# ---------------------------------------------------------------
# 4. Centrality measures
# ---------------------------------------------------------------
degree_c = nx.degree_centrality(G)
betweenness_c = nx.betweenness_centrality(G)
closeness_c = nx.closeness_centrality(G)
eigenvector_c = nx.eigenvector_centrality(G, max_iter=1000)

centrality_df = pd.DataFrame({
    "node": list(G.nodes()),
    "degree_centrality": [degree_c[n] for n in G.nodes()],
    "betweenness_centrality": [betweenness_c[n] for n in G.nodes()],
    "closeness_centrality": [closeness_c[n] for n in G.nodes()],
    "eigenvector_centrality": [eigenvector_c[n] for n in G.nodes()],
}).sort_values("degree_centrality", ascending=False)
centrality_df.to_csv("centrality_results.csv", index=False)

print("\nTop 5 nodes by degree centrality:")
print(centrality_df[["node", "degree_centrality"]].head(5).to_string(index=False))
print("\nTop 5 nodes by betweenness centrality (bridges/brokers):")
print(centrality_df.sort_values("betweenness_centrality", ascending=False)
      [["node", "betweenness_centrality"]].head(5).to_string(index=False))

with open("graph_results.txt", "w") as f:
    f.write("GRAPH TRAVERSAL & CENTRALITY ANALYSIS\n")
    f.write("=" * 50 + "\n")
    f.write(f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}\n")
    f.write(f"Connected: {nx.is_connected(G)}\n")
    f.write(f"Start node: {start_node}\n\n")
    f.write(f"BFS order ({len(bfs_order)} nodes): {bfs_order}\n\n")
    f.write(f"DFS order ({len(dfs_order)} nodes): {dfs_order}\n\n")
    f.write("Top 5 nodes by degree centrality:\n")
    f.write(centrality_df[["node", "degree_centrality"]].head(5).to_string(index=False))
    f.write("\n\nTop 5 nodes by betweenness centrality:\n")
    f.write(centrality_df.sort_values("betweenness_centrality", ascending=False)
            [["node", "betweenness_centrality"]].head(5).to_string(index=False))

# ---------------------------------------------------------------
# 5. Visualizations
# ---------------------------------------------------------------
pos = nx.spring_layout(G, seed=42, k=0.5)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

ax = axes[0, 0]
nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.2)
nx.draw_networkx_edges(G, pos, edgelist=bfs_edges, ax=ax, edge_color="#1f77b4", width=2)
node_colors = ["#1f77b4" if n in bfs_order else "#cccccc" for n in G.nodes()]
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=150, ax=ax)
nx.draw_networkx_nodes(G, pos, nodelist=[start_node], node_color="red", node_size=250, ax=ax)
ax.set_title(f"BFS Traversal Tree from {start_node}")
ax.axis("off")

ax = axes[0, 1]
nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.2)
nx.draw_networkx_edges(G, pos, edgelist=dfs_edges, ax=ax, edge_color="#2ca02c", width=2)
node_colors = ["#2ca02c" if n in dfs_order else "#cccccc" for n in G.nodes()]
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=150, ax=ax)
nx.draw_networkx_nodes(G, pos, nodelist=[start_node], node_color="red", node_size=250, ax=ax)
ax.set_title(f"DFS Traversal Tree from {start_node}")
ax.axis("off")

ax = axes[1, 0]
sizes = [3000 * degree_c[n] + 50 for n in G.nodes()]
colors = [degree_c[n] for n in G.nodes()]
nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.2)
nodes = nx.draw_networkx_nodes(G, pos, node_size=sizes, node_color=colors, cmap=cm.viridis, ax=ax)
ax.set_title("Degree Centrality (node size/color = importance)")
ax.axis("off")
plt.colorbar(nodes, ax=ax, fraction=0.046)

ax = axes[1, 1]
sizes = [3000 * betweenness_c[n] + 50 for n in G.nodes()]
colors = [betweenness_c[n] for n in G.nodes()]
nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.2)
nodes = nx.draw_networkx_nodes(G, pos, node_size=sizes, node_color=colors, cmap=cm.plasma, ax=ax)
ax.set_title("Betweenness Centrality (bridges/brokers)")
ax.axis("off")
plt.colorbar(nodes, ax=ax, fraction=0.046)

plt.tight_layout()
plt.savefig("graph_plots.png", dpi=150)
print("\nSaved: social_network_edges.csv, centrality_results.csv, graph_results.txt, graph_plots.png")

Graph: 40 nodes, 76 edges
Is connected: True
Start node for traversal (highest degree): User_1

BFS traversal order (40 nodes reached):
['User_1', 'User_2', 'User_3', 'User_4', 'User_6', 'User_8', 'User_10', 'User_11', 'User_12', 'User_14', 'User_15', 'User_18', 'User_19', 'User_20', 'User_21', 'User_23', 'User_24', 'User_26', 'User_29', 'User_33', 'User_37', 'User_39', 'User_5', 'User_7', 'User_9', 'User_16', 'User_22', 'User_28', 'User_17', 'User_13', 'User_25', 'User_31', 'User_34', 'User_35', 'User_40', 'User_27', 'User_36', 'User_30', 'User_32', 'User_38']
Max BFS hop distance from User_1: 3

DFS traversal order (40 nodes reached):
['User_1', 'User_2', 'User_4', 'User_9', 'User_11', 'User_33', 'User_15', 'User_21', 'User_27', 'User_14', 'User_5', 'User_3', 'User_17', 'User_22', 'User_26', 'User_36', 'User_25', 'User_6', 'User_12', 'User_13', 'User_38', 'User_7', 'User_10', 'User_30', 'User_32', 'User_40', 'User_23', 'User_31', 'User_34', 'User_35', 'User_20', 'User_28', 'User_16',